##### ktor-client, dataframe, kandy, org.json.XML, org.sqlite.JDBC
* Convert XML data collected via the ktor client to JSON type and load it using DataFrame.readJson.
* Load the SQLite table using DataFrame.readSqlQuery.
* Then, join the two dataframes and create a chart using Kandy.

In [128]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [129]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [ ]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(36)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

In [131]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [132]:
@file:DependsOn("org.json:json:20250107")

In [133]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [134]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Comparable<*>,2786,1468,0,0.260000,72,null,null,null,null,null,null,null
rtmWqChpla,Comparable<*>,2786,1605,0,,230,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2786,1,0,,2786,null,null,,,,,
rtmWqWtchStaCd,String,2786,14,0,SEA5001,231,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2786,2786,0,1,1,1393.500000,804.393250,1,696.916667,1393.500000,2090.083333,2786
rtmWqTu,Comparable<*>,2786,266,0,5,237,null,null,null,null,null,null,null
ph,Comparable<*>,2786,318,0,7.480000,43,null,null,null,null,null,null,null
rtmWqSlnty,Number,2786,2532,0,0.260000,19,21.364922,10.600254,0.000000,11.774000,26.268999,29.525999,34.032001
rtmWqCndctv,Number,2786,2628,0,52.230000,6,33.454404,15.899362,0.000000,19.778000,39.611000,44.791000,54.451000
rtmWqWtchDtlDt,String,2786,242,0,2026-07-21 09:40:00.0,14,null,null,2026-07-21 09:40:00.0,2026-07-21 17:15:00.0,2026-07-22 00:45:00.0,2026-07-22 08:30:00.0,2026-07-22 15:45:00.0


In [136]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert {
    rtmWqDoxn and rtmWqChpla and rtmWqSlnty and rtmWqCndctv and rtmWtchWtem and ph
}.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else  value.toDouble()
}.convert {
    rtmWqTu
}.with{
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0 else value.toInt()
}

df.schema()

rtmWqDoxn: Double
rtmWqChpla: Double
rtmWqWtchStaCd: String
num: Int
rtmWqTu: Int
ph: Double
rtmWqSlnty: Double
rtmWqCndctv: Double
rtmWqWtchDtlDt: LocalDateTime
rtmWtchWtem: Double

In [137]:
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2786,1460,0,0.260000,72,5.956187,3.015395,0.000000,4.979167,5.890000,6.920000,23.021000
rtmWqChpla,Double,2786,1604,0,0.000000,233,5.784254,6.074922,0.000000,1.415500,3.150000,8.237250,39.261000
rtmWqWtchStaCd,String,2786,14,0,SEA5001,231,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2786,2786,0,1,1,1393.500000,804.393250,1,696.916667,1393.500000,2090.083333,2786
rtmWqTu,Int,2786,265,0,5,237,36.828069,67.614688,0,4.000000,11.500000,35.000000,552
ph,Double,2786,313,0,7.480000,43,7.721637,0.450122,0.000000,7.440000,7.630000,8.000000,9.080000
rtmWqSlnty,Double,2786,2531,0,0.260000,19,21.364922,10.600254,0.000000,11.776750,26.269500,29.526083,34.032001
rtmWqCndctv,Double,2786,2627,0,52.230000,6,33.454404,15.899362,0.000000,19.783500,39.621000,44.791250,54.451000
rtmWqWtchDtlDt,LocalDateTime,2786,242,0,2026-07-21T09:40,14,null,null,2026-07-21T09:40,2026-07-21T17:15,2026-07-22T00:45,2026-07-22T08:30,2026-07-22T15:45
rtmWtchWtem,Double,2786,839,0,26.260000,16,26.596059,2.493309,0.000000,24.969999,26.500000,28.700001,32.029999


In [138]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [139]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-21T09:40,7.780000,2.650000,NEP1002,8,7.860000,0.460000,0.933000,29.379999
2,2026-07-21T09:40,3.170000,0.000000,SEA1005,25,7.290000,2.567000,4.810000,26.000000
3,2026-07-21T09:40,6.280000,2.090000,NEP2002,13,7.910000,29.209999,44.493000,24.250000
4,2026-07-21T09:40,5.610000,7.880000,SEA7002,17,7.490000,26.634001,38.808000,21.610001
5,2026-07-21T09:40,0.260000,0.800000,SEA5002,66,7.130000,24.902000,39.217000,27.990000


In [140]:
USE {
    dependencies {
        implementation("org.xerial:sqlite-jdbc:3.49.1.0")
        implementation("ch.qos.logback:logback-classic:1.5.12")
    }
}

In [141]:
import java.sql.Connection
import java.sql.DriverManager

Class.forName("org.sqlite.JDBC")
val connection = DriverManager.getConnection("jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite")

In [142]:
val sqlStmt = "SELECT * FROM OWQObservatory"
val df_list = DataFrame.readSqlQuery(connection, sqlStmt)
df_list.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
sta_code,String,19,19,0,SEA1002,1,null,null,NEP1001,NEP3001,SEA1301,SEA3003,SEA7002
sta_name,String,19,19,0,시화조력,1,null,null,광양망덕,낙동명지,부산수영,영산목포,천수만
ocean_division,String,19,2,0,특별관리해역,12,null,null,특별관리해역,특별관리해역,특별관리해역,하구 및 만,하구 및 만
lon,Double,19,19,0,126.611000,1,127.588263,1.113453,126.366000,126.540167,127.605000,128.615167,129.387000
lat,Double,19,18,0,35.802000,2,35.687263,0.981615,34.782000,34.990667,35.211000,35.947833,37.731000


In [143]:
val joinedDf = removedDf.join(df_list) { 관측정점코드 match right.sta_code }

In [144]:
joinedDf
    .select{  일시 and 클로로필 and sta_name   }
//    .convert{클로로필}.toDouble()
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(클로로필) {axis.name ="클로로필"}
        line{
            color(sta_name){
             //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="aNKpVV" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("aNKpVV");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"sta_name":["낙동명지","시화반월","영산목포","울산매암","광양초남","시화조력","새만금","광양망덕","금강하구","광양적량","영산영암","마산봉암","천수만","울산매암","광양망덕","새만금","영산목포","시화반월","광양적량","금강하구","광양초남","광양망덕","울산매암","금강하구","영산영암","새만금","영산목포","광양적량","시화반월","낙동명지","마산봉암","천수만","광양초남","마산봉암","광양적량","영산목포","울산매암","금강하구","광양초남","광양망덕","영산영암","시화반월","새만금","영산목포","영산영암","광양초남","울산매암","시화반월","천수만","낙동명지","마산봉암","광양망덕","시화조력","금강하구","광양적량","광양망덕","마산봉암","영산목포","광양초남","영산영암","금강하구","시화반월","광양적량","울산매암","새만금","울산매암","시화반월","천수만","마산봉암","새만금","광양적량","낙동명지","영산영암","금강하구","광양망덕","영산목포","광양초남","영산영암","금강하구","마산봉암","영산목포","광양망덕","낙동명지","광양초남","시화반월","광양적량","낙동명지","시화조력","마산봉암","광양망덕","새만금","울산매암","영산목포","영산영암","천수만","시화반월","금강하구","마산봉암","새만금","영산영암","광양망덕","광양적량","울산매암","시화반월","영산목포","금강하구","광양망덕","시화반월","천수만","시화조력","마산봉암","금강하구","울산매암","광양초남","영산목포","광양적량","새만금","영산영암","광양망덕","금강하구","시화반월","마산봉암","광양초남","영산목포","영산영암","마산봉암","낙동명지","광양적량","울산매암","영산목포","금강하구","광양망덕","시화조력","영산영암","시화반월","새만금","영산목포","새만금","시화반월","금강하구","영산영암","광양적량","울산매암","울산매암","마산봉암","시화반월","광양초남","영산영암","금강하구","새만금","영산목포","광양망덕","천수만","광양적량","시화조력","광양적량","광양망덕","영산영암","영산목포","시화반월","새만금","금강하구","광양초남","시화반월","광양초남","광양망덕","영산목포","영산영암","금강하구","광양적량","시화조력","새만금","낙동명지","울산매암","새만금","광양적량","영산영암","금강하구","영산목포","광양초남","울산매암","시화반월","광양초남","낙동명지","울산매암","영산영암","새만금","광양적량","영산목포","광양망덕","금강하구","마산봉암","시화조력","천수만","시화반월","영산목포","마산봉암","광양적량","금강하구","영산영암","새만금","광양초남","광양망덕","울산매암","시화반월","천수만","영산목포","영산영암","마산봉암","울산매암","시화조력","광양초남","새만금","광양망덕","금강하구","광양적량","시화반월","낙동명지","마산봉암","영산목포","광양적량","영산영암","새만금","시화반월","광양망덕","울산매암","금강하구","천수만","울산매암","금강하구","새만금","낙동명지","광양망덕","광양적량","마산봉암","시화조력","광양초남","시화반월","영산영암","마산봉암","울산매암","광양초남","광양적량","새만금","시화반월","영산영암","광양망덕","시화조력","울산매암","시화반월","영산목포","광양초남","금강하구","새만금","광양망덕","낙동명지","광양적량","천수만","마산봉암","영산영암","광양망덕","새만금","영산영암","시화반월","금강하구","영산목포","마산봉암","울산매암","광양적량","영산영암","울산매암","새만금","시화반월","낙동명지","천수만","광양망덕","광양적량","금강하구","광양초남","시화조력","마산봉암","영산목포","마산봉암","영산목포","울산매암","시화반월","광양초남","새만금","광양적량","광양망덕","시화조력","마산봉암","낙동명지","천수만","광양초남","광양망덕","광양적량","영산영암","울산매암","영산목포","새만금","시화반월","울산매암","광양적량","영산영암","마산봉암","새만금","광양망덕","시화반월","광양초남","영산영암","시화조력","광양적량","시화반월","새만금","영산목포","광양망덕","울산매암","천수만","금강하구","낙동명지","마산봉암","광양초남","시화반월","울산매암","광양적량","광양망덕","영산목포","영산영암","새만금","마산봉암","금강하구","광양초남","울산매암","광양망덕","낙동명지","광양적량","새만금","금강하구","영산목포","시화반월","시화조력","영산영암","마산봉암","광양망덕","광양초남","새만금","마산봉암","광양적량","시화반월","금강하구","울산매암","광양적량","영산영암","시화조력","금강하구","새만금","마산봉암","영산목포","낙동명지","천수만","광양초남","광양망덕","울산매암","시화반월","마산봉암","금강하구","영산목포","새만금","영산영암","울산매암","광양적량","광양초남","시화반월","광양망덕","새만금","천수만","광양망덕","울산매암","광양적량","금강하구","시화조력","시화반월","마산봉암","광양초남","영산영암","영산목포","낙동명지","광양망덕","영산목포","새만금","광양적량","마산봉암","금강하구","광양초남","울산매암","낙동명지","광양적량","광양초남","광양망덕","마산봉암","시화반월","시화조력","금강하구","새만금","영산목포","울산매암","천수만","영산영암","마산봉암","울산매암","영산영암","시화반월","광양적량","새만금","영산목포","광양초남","금강하구","광양망덕","마산봉암","낙동명지","시화조력","금강하구","영산목포","광양초남","울산매암","시화반월","새만금","광양망덕","영산영암","광양적량","광양초남","마산봉암","금강하구","광양망덕","광양적량","영산목